# RAG чатбот для документів

In [ ]:
import io
import re
import sqlite3
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import requests
from docx import Document
from IPython.display import Markdown, clear_output, display
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4
SIMILARITY_THRESHOLD = 0.35
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
OLLAMA_MODEL = "qwen2.5:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"
DB_PATH = "chat_history.db"

SYSTEM_PROMPT = """Ти — AI-асистент для роботи з документами.
Відповідай українською мовою.
Використовуй лише інформацію з наданого контексту.
Не вигадуй факти та не використовуй зовнішні знання.
Якщо відповіді немає в контексті, прямо повідом про це.
Відповідай коротко та по суті."""


In [ ]:
def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS conversations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                role TEXT NOT NULL,
                content TEXT NOT NULL,
                sources TEXT DEFAULT '',
                timestamp TEXT NOT NULL
            )
        """)


def save_message(role, content, sources=""):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO conversations (role, content, sources, timestamp) VALUES (?, ?, ?, ?)",
            (role, content, sources, datetime.now().isoformat(timespec="seconds")),
        )


def get_history():
    with sqlite3.connect(DB_PATH) as conn:
        return conn.execute(
            "SELECT role, content, sources, timestamp FROM conversations ORDER BY id"
        ).fetchall()


def clear_history():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("DELETE FROM conversations")


def export_history():
    lines = []
    for role, content, sources, timestamp in get_history():
        name = "Користувач" if role == "user" else "Асистент"
        lines.extend([f"[{timestamp}] {name}:", content])
        if sources:
            lines.append(sources)
        lines.append("-" * 40)
    return "\n".join(lines)


init_db()


In [ ]:
def extract_pdf(data):
    reader = PdfReader(io.BytesIO(data))
    return " ".join(page.extract_text() or "" for page in reader.pages)


def extract_docx(data):
    doc = Document(io.BytesIO(data))
    return " ".join(paragraph.text for paragraph in doc.paragraphs)


def extract_txt(data):
    try:
        return data.decode("utf-8-sig")
    except UnicodeDecodeError:
        return data.decode("cp1251", errors="ignore")



splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
)


def split_chunks(documents):
    chunks = []
    for filename, text in documents:
        text = re.sub(r"\s+", " ", text).strip() # One space
        for number, part in enumerate(splitter.split_text(text), 1):
            chunks.append({"text": part, "filename": filename, "chunk": number})
    return chunks


In [ ]:
model = SentenceTransformer(MODEL_NAME)


def embed(texts):
    return model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
    )


def search(query, chunks, embeddings, selected):
    indexes = [
        i for i, chunk in enumerate(chunks)
        if chunk["filename"] in selected
    ]
    if not indexes:
        return []

    query_embedding = model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_embedding, embeddings[indexes])[0]
    ranked = np.argsort(scores)[::-1]

    return [
        (chunks[indexes[i]], float(scores[i]))
        for i in ranked
        if scores[i] >= SIMILARITY_THRESHOLD
    ][:TOP_K]


def answer(query, results):
    context = "\n\n".join(
        f"[{chunk['filename']}, фрагмент {chunk['chunk']}]\n{chunk['text']}"
        for chunk, _ in results
    )
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "system": SYSTEM_PROMPT,
            "prompt": f"Контекст:\n{context}\n\nПитання: {query}",
            "stream": False,
        },
        timeout=180,
    )
    response.raise_for_status()
    return response.json()["response"].strip()


In [ ]:
CHUNKS = []
EMBEDDINGS = None

uploader = widgets.FileUpload(accept=".pdf,.docx,.txt", multiple=True)
btn_process = widgets.Button(description="Обробити")
file_checks = widgets.VBox()
query_box = widgets.Text(
    placeholder="Питання",
    layout=widgets.Layout(width="420px"),
)
btn_ask = widgets.Button(description="Запитати")
btn_export = widgets.Button(description="Експорт TXT")
btn_clear = widgets.Button(description="Очистити історію", button_style="danger")
history_output = widgets.Output()
status_output = widgets.Output()


def selected_files():
    return [checkbox.description for checkbox in file_checks.children if checkbox.value]


def render_history():
    with history_output:
        clear_output(wait=True)
        for role, content, sources, timestamp in get_history():
            name = "Користувач" if role == "user" else "Асистент"
            display(Markdown(f"**{name}** · {timestamp}\n\n{content}"))
            if sources:
                display(Markdown(sources))
            display(Markdown("---"))


def on_process(_):
    global CHUNKS, EMBEDDINGS
    CHUNKS = []
    EMBEDDINGS = None
    documents = []

    items = uploader.value
    files = list(items.items()) if isinstance(items, dict) else [
        (item["name"], item) for item in items
    ]

    with status_output:
        clear_output()
        if not files:
            print("Оберіть документи.")
            return

        for name, info in files:
            data = info["content"] if isinstance(info, dict) else info.content
            data = bytes(data)
            if name.lower().endswith(".pdf"):
                text = extract_pdf(data)
            elif name.lower().endswith(".docx"):
                text = extract_docx(data)
            elif name.lower().endswith(".txt"):
                text = extract_txt(data)
            else:
                continue

            if text.strip():
                documents.append((name, text))

        CHUNKS = split_chunks(documents)
        if not CHUNKS:
            file_checks.children = ()
            print("Текст не знайдено.")
            return

        EMBEDDINGS = embed([chunk["text"] for chunk in CHUNKS])
        names = list(dict.fromkeys(chunk["filename"] for chunk in CHUNKS))
        file_checks.children = tuple(
            widgets.Checkbox(value=True, description=name, indent=False)
            for name in names
        )
        print(f"Готово: {len(CHUNKS)} фрагментів.")


def on_ask(_):
    query = query_box.value.strip()
    with status_output:
        clear_output()
        if not query:
            print("Введіть питання.")
            return
        if EMBEDDINGS is None:
            print("Спочатку обробіть документи.")
            return

        results = search(query, CHUNKS, EMBEDDINGS, selected_files())
        if results:
            response = answer(query, results)
            source_parts = ["### Джерела"]
            for i, (chunk, score) in enumerate(results, 1):
                source_parts.append(
                    f"**{i}.** `{chunk['filename']}`, фрагмент {chunk['chunk']} "
                    f"(схожість: {score:.3f})\n\n"
                    f"> {chunk['text']}"
                )
            sources = "\n\n".join(source_parts)
        else:
            response = "У завантажених документах не знайдено інформації для відповіді."
            sources = ""

        save_message("user", query)
        save_message("assistant", response, sources)
        display(Markdown(response))
        if sources:
            display(Markdown(sources))

    render_history()
    query_box.value = ""


def on_export(_):
    path = Path(f"chat_history_{datetime.now():%Y%m%d_%H%M%S}.md")
    path.write_text(export_history(), encoding="utf-8")
    with status_output:
        clear_output()
        print(f"Збережено: {path}")


def on_clear(_):
    clear_history()
    render_history()
    with status_output:
        clear_output()
        print("Історію очищено.")


btn_process.on_click(on_process)
btn_ask.on_click(on_ask)
btn_export.on_click(on_export)
btn_clear.on_click(on_clear)

display(uploader, btn_process)
display(Markdown("### Документи для пошуку"))
display(file_checks)
display(query_box, widgets.HBox([btn_ask, btn_export, btn_clear]))
display(status_output)
display(Markdown("## Історія"))
display(history_output)
render_history()
